# 01 — PlantVillage Classification Data Preparation and Bias-Audit Setup

This classification study will go beyond a normal CNN benchmark.

Planned experiments:

1. **Standard RGB EfficientNet-B0 baseline**
2. **Leaf-only EfficientNet-B0 teacher/baseline**
3. **Proposed mask-guided counterfactual consistency distillation**
4. **Robustness + shortcut-learning comparison**

The proposed student will ultimately require only an ordinary RGB image at inference time.

This notebook is CPU-friendly and reuses the finalized segmentation data lineage:

- 52,969 samples
- 42,371 train
- 5,300 validation
- 5,298 test
- 38 classes

The same validated leaf masks are reused to create four views:

```text
Original RGB
Leaf-only
Background-only
Counterfactual background
```

Do not claim the method is "first ever" until the final systematic literature review is complete.


# STEP 1 — Install and Import

In [ ]:
!pip -q install kagglehub

import json, hashlib, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import kagglehub
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Seed:", SEED)


# STEP 2 — Download PlantVillage and Detect Root

In [ ]:
download_path = Path(
    kagglehub.dataset_download(
        "abdallahalidev/plantvillage-dataset"
    )
)

def find_dataset_root(base):
    required = {"color","grayscale","segmented"}
    candidates = [base] + [p for p in base.rglob("*") if p.is_dir()]
    for path in candidates:
        try:
            folders = {x.name for x in path.iterdir() if x.is_dir()}
            if required.issubset(folders):
                return path
        except PermissionError:
            pass
    return None

dataset_root = find_dataset_root(download_path)

assert dataset_root is not None
assert (dataset_root/"color").exists()
assert (dataset_root/"segmented").exists()

print("Dataset root:", dataset_root)
print("✅ PlantVillage structure found.")


# STEP 3 — Mount Drive and Create Classification Project

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SEGMENTATION_PROJECT_DIR = Path(
    "/content/drive/MyDrive/PlantVillage_Segmentation"
)

CLASSIFICATION_PROJECT_DIR = Path(
    "/content/drive/MyDrive/PlantVillage_Classification"
)

ARTIFACT_DIR = CLASSIFICATION_PROJECT_DIR/"artifacts"
VISUAL_DIR = CLASSIFICATION_PROJECT_DIR/"visuals"

for p in [CLASSIFICATION_PROJECT_DIR, ARTIFACT_DIR, VISUAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Classification project:", CLASSIFICATION_PROJECT_DIR)


# STEP 4 — Load Final QC-Approved Manifest

In [ ]:
SEGMENTATION_MANIFEST = SEGMENTATION_PROJECT_DIR/"final_segmentation_manifest.csv"
SEGMENTATION_CONFIG = SEGMENTATION_PROJECT_DIR/"segmentation_preprocessing_config.json"

assert SEGMENTATION_MANIFEST.exists()
assert SEGMENTATION_CONFIG.exists()

source_df = pd.read_csv(SEGMENTATION_MANIFEST)

with open(SEGMENTATION_CONFIG,"r") as f:
    segmentation_config = json.load(f)

print("Total:", len(source_df))
print(source_df["split"].value_counts())
print("Classes:", source_df["class"].nunique())

assert len(source_df) == 52969
assert (source_df["split"]=="train").sum() == 42371
assert (source_df["split"]=="val").sum() == 5300
assert (source_df["split"]=="test").sum() == 5298
assert source_df["class"].nunique() == 38

BLACK_THRESHOLD = int(segmentation_config["black_threshold"])
assert BLACK_THRESHOLD == 15

print("✅ Final source manifest verified.")


## Why reuse this split?

We do **not** create another random classification split.

That preserves:

- leakage control,
- identical data lineage across tasks,
- reproducibility,
- direct reuse of the already validated leaf masks.


# STEP 5 — Build Stable Class Mapping and Classification Manifest

In [ ]:
class_names = sorted(source_df["class"].unique())
class_to_id = {name:i for i,name in enumerate(class_names)}
id_to_class = {i:name for name,i in class_to_id.items()}

classification_df = source_df.copy().reset_index(drop=True)
classification_df["class_id"] = classification_df["class"].map(class_to_id).astype(int)
classification_df["sample_id"] = classification_df["color_rel"].apply(
    lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest()[:16]
)

assert len(class_names) == 38
assert classification_df["sample_id"].duplicated().sum() == 0

print(classification_df[[
    "sample_id","class","class_id","split","color_rel","segmented_rel"
]].head())


# STEP 6 — Recreate Final Leaf Mask

In [ ]:
def create_leaf_mask(segmented_rgb, target_h, target_w, threshold=15):
    segmented_rgb = np.asarray(segmented_rgb, dtype=np.uint8)
    mask = (np.max(segmented_rgb, axis=2) > threshold).astype(np.uint8)

    if mask.shape != (target_h,target_w):
        mask = cv2.resize(
            mask,
            (target_w,target_h),
            interpolation=cv2.INTER_NEAREST
        )

    return mask


# STEP 7 — Define the Four Classification Views

In [ ]:
# Neutral background approximately equal to ImageNet mean RGB × 255.
NEUTRAL_RGB = np.array([124,116,104], dtype=np.uint8)

def make_leaf_only(image, mask):
    out = np.empty_like(image)
    out[:] = NEUTRAL_RGB
    out[mask==1] = image[mask==1]
    return out

def make_background_only(image, mask):
    out = image.copy()
    out[mask==1] = NEUTRAL_RGB
    return out

def make_clean_donor_background(donor_image, donor_mask):
    out = donor_image.copy()

    bg_pixels = out[donor_mask==0]

    fill_rgb = (
        np.median(bg_pixels, axis=0).astype(np.uint8)
        if len(bg_pixels) > 0
        else NEUTRAL_RGB
    )

    out[donor_mask==1] = fill_rgb
    out = cv2.GaussianBlur(out,(15,15),0)

    return out

def make_counterfactual(target_image, target_mask, donor_background):
    if donor_background.shape[:2] != target_image.shape[:2]:
        donor_background = cv2.resize(
            donor_background,
            (target_image.shape[1], target_image.shape[0]),
            interpolation=cv2.INTER_LINEAR
        )

    out = donor_background.copy()
    out[target_mask==1] = target_image[target_mask==1]
    return out


## Why these views?

**Original RGB:** ordinary classification input.

**Leaf-only:** removes visible background cues and will train the leaf-focused teacher.

**Background-only:** removes the leaf. If a classifier still performs surprisingly well, that is evidence of shortcut/background bias.

**Counterfactual background:** keeps the same target leaf but changes only the background. The proposed student will be trained to remain consistent under this intervention.


# STEP 8 — Create Fixed Cross-Class Background Donor Mapping

In [ ]:
rng = np.random.default_rng(SEED)
classification_df["counterfactual_partner_id"] = None

for split_name in ["train","val","test"]:

    split_indices = classification_df.index[
        classification_df["split"]==split_name
    ].to_numpy()

    split_classes = classification_df.loc[
        split_indices,"class_id"
    ].to_numpy()

    by_class = {
        cid: split_indices[split_classes==cid]
        for cid in range(38)
    }

    for idx in tqdm(split_indices, desc=f"Partner mapping: {split_name}"):

        source_class = int(classification_df.at[idx,"class_id"])

        candidate_classes = [
            cid for cid in range(38)
            if cid != source_class and len(by_class[cid]) > 0
        ]

        donor_class = int(rng.choice(candidate_classes))
        donor_idx = int(rng.choice(by_class[donor_class]))

        classification_df.at[
            idx,"counterfactual_partner_id"
        ] = classification_df.at[donor_idx,"sample_id"]

assert classification_df["counterfactual_partner_id"].isna().sum() == 0

sample_to_row = classification_df.set_index("sample_id")

violations = 0

for row in classification_df.sample(
    n=500,
    random_state=SEED
).itertuples(index=False):

    partner = sample_to_row.loc[row.counterfactual_partner_id]

    if (
        partner["split"] != row.split
        or int(partner["class_id"]) == int(row.class_id)
    ):
        violations += 1

print("Partner QC violations:", violations)
assert violations == 0
print("✅ Same-split, different-class partner mapping complete.")


# STEP 9 — Class Distribution Audit

In [ ]:
distribution = (
    classification_df.groupby(
        ["class_id","class","split"]
    )
    .size()
    .reset_index(name="count")
)

pivot_distribution = distribution.pivot_table(
    index=["class_id","class"],
    columns="split",
    values="count",
    fill_value=0
).reset_index()

display(pivot_distribution)

total_per_class = classification_df["class"].value_counts()

print("Min samples/class:", total_per_class.min())
print("Max samples/class:", total_per_class.max())


# STEP 10 — Save Class Distribution Plot

In [ ]:
counts = classification_df["class"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(15,7))
counts.plot(kind="bar")
plt.title("PlantVillage Classification Class Distribution")
plt.ylabel("Samples")
plt.xlabel("Class")
plt.xticks(rotation=90,fontsize=7)
plt.grid(axis="y",alpha=0.3)
plt.tight_layout()

CLASS_DISTRIBUTION_FIG = VISUAL_DIR/"class_distribution.png"

plt.savefig(
    CLASS_DISTRIBUTION_FIG,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

print("Saved:", CLASS_DISTRIBUTION_FIG)


# STEP 11 — Visual Bias-Audit Examples

In [ ]:
preview_rows = classification_df.sample(
    n=6,
    random_state=SEED
).reset_index(drop=True)

fig, axes = plt.subplots(
    len(preview_rows),
    4,
    figsize=(16,4*len(preview_rows))
)

for r,row in preview_rows.iterrows():

    target = np.array(
        Image.open(dataset_root/row["color_rel"]).convert("RGB")
    )

    target_seg = np.array(
        Image.open(dataset_root/row["segmented_rel"]).convert("RGB")
    )

    target_mask = create_leaf_mask(
        target_seg,
        target.shape[0],
        target.shape[1],
        BLACK_THRESHOLD
    )

    donor_row = sample_to_row.loc[
        row["counterfactual_partner_id"]
    ]

    donor = np.array(
        Image.open(dataset_root/donor_row["color_rel"]).convert("RGB")
    )

    donor_seg = np.array(
        Image.open(dataset_root/donor_row["segmented_rel"]).convert("RGB")
    )

    donor_mask = create_leaf_mask(
        donor_seg,
        donor.shape[0],
        donor.shape[1],
        BLACK_THRESHOLD
    )

    donor_bg = make_clean_donor_background(
        donor,
        donor_mask
    )

    views = [
        target,
        make_leaf_only(target,target_mask),
        make_background_only(target,target_mask),
        make_counterfactual(target,target_mask,donor_bg)
    ]

    titles = [
        "Original RGB",
        "Leaf-only",
        "Background-only",
        "Counterfactual BG"
    ]

    for c,(img,title) in enumerate(zip(views,titles)):
        axes[r,c].imshow(img)
        axes[r,c].set_title(
            f"{title}\n{row['class']}",
            fontsize=8
        )
        axes[r,c].axis("off")

plt.tight_layout()

BIAS_AUDIT_FIG = VISUAL_DIR/"classification_bias_audit_views.png"

plt.savefig(
    BIAS_AUDIT_FIG,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

print("Saved:", BIAS_AUDIT_FIG)


## Stop and inspect STEP 11 carefully

A good counterfactual example should show:

- the **same target leaf**,
- a visibly changed background,
- no second donor leaf,
- no obvious broken mask around the target.

If this looks wrong, do not start model training yet.


# STEP 12 — Save Final Classification Artifacts

In [ ]:
FINAL_MANIFEST = (
    CLASSIFICATION_PROJECT_DIR /
    "final_classification_manifest.csv"
)

CLASS_MAP_PATH = (
    CLASSIFICATION_PROJECT_DIR /
    "classification_class_map.json"
)

DISTRIBUTION_CSV = (
    ARTIFACT_DIR /
    "class_distribution_by_split.csv"
)

classification_df.to_csv(
    FINAL_MANIFEST,
    index=False
)

with open(CLASS_MAP_PATH,"w") as f:
    json.dump(
        {
            "class_to_id": class_to_id,
            "id_to_class": {
                str(k):v
                for k,v in id_to_class.items()
            }
        },
        f,
        indent=4
    )

pivot_distribution.to_csv(
    DISTRIBUTION_CSV,
    index=False
)

print("Saved:", FINAL_MANIFEST)
print("Saved:", CLASS_MAP_PATH)
print("Saved:", DISTRIBUTION_CSV)


# STEP 13 — Save Research Configuration

In [ ]:
classification_config = {
    "task":
        "38-class PlantVillage disease classification",

    "research_direction":
        "mask-guided counterfactual consistency distillation",

    "source_samples":
        52969,

    "train_samples":
        42371,

    "validation_samples":
        5300,

    "test_samples":
        5298,

    "classes":
        38,

    "mask_threshold":
        BLACK_THRESHOLD,

    "neutral_background_rgb":
        NEUTRAL_RGB.tolist(),

    "counterfactual_partner_rule":
        "same split, different class, seed 42",

    "planned_models": [
        "Standard RGB EfficientNet-B0 baseline",
        "Leaf-only EfficientNet-B0 teacher/baseline",
        "Mask-guided counterfactual consistency distillation student"
    ],

    "planned_metrics": [
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1"
    ],

    "planned_robustness_tests": [
        "original RGB",
        "leaf-only",
        "background-only diagnostic",
        "counterfactual background"
    ],

    "novelty_note":
        (
            "Do not claim first-ever use until a systematic "
            "literature review verifies the exact method combination."
        )
}

CONFIG_PATH = (
    CLASSIFICATION_PROJECT_DIR /
    "classification_preprocessing_config.json"
)

with open(CONFIG_PATH,"w") as f:
    json.dump(
        classification_config,
        f,
        indent=4
    )

print(
    json.dumps(
        classification_config,
        indent=2
    )
)


# STEP 14 — Final Verification

In [ ]:
required = [
    FINAL_MANIFEST,
    CLASS_MAP_PATH,
    CONFIG_PATH,
    DISTRIBUTION_CSV,
    CLASS_DISTRIBUTION_FIG,
    BIAS_AUDIT_FIG
]

for p in required:
    print(
        "✅" if Path(p).exists() else "❌",
        p
    )
    assert Path(p).exists()

print()
print(classification_df["split"].value_counts())
print("Classes:", classification_df["class"].nunique())

assert len(classification_df) == 52969
assert classification_df["class"].nunique() == 38

print(
    "\n✅ CLASSIFICATION DATA PREPARATION COMPLETE."
)


# Next notebooks

After this notebook is visually verified:

```text
02_PlantVillage_Standard_RGB_Classification.ipynb
03_PlantVillage_Leaf_Only_Teacher_Classification.ipynb
04_PlantVillage_Mask_Guided_Counterfactual_Consistency_Distillation.ipynb
05_PlantVillage_Classification_Robustness_and_Model_Comparison.ipynb
```

The research question is not only:

> Which classifier has the highest ordinary PlantVillage accuracy?

It is also:

> Which classifier remains stable when the background changes, and how much class information can be recovered from the background alone?
